In [20]:
import os
import numpy as np
import random
import gzip
import json
import torch
from tqdm import tqdm

In [21]:
content_dir = "habitat-lab/data/datasets/PersONAL/active_new/val/test_baselines/easy/content"
content_dir = "habitat-lab/data/datasets/PersONAL/active_new/val/test_baselines/medium/content"
content_dir = "habitat-lab/data/datasets/PersONAL/active_new/val/test_baselines/hard/content"

len(os.listdir(content_dir))

31

In [22]:
descr = {}
img_rand = torch.randn((3, 640, 480))

for f in os.listdir(content_dir):

    f_path = os.path.join(content_dir, f)

    with gzip.open(f_path, "r") as i:
        info = json.load(i)

    info = info["episodes"]
    for ep_info in info:
        scene_id, ep_id = ep_info["scene_id"].split("/")[-1].split(".")[0], \
                            ep_info["episode_id"]

        # descr[f"{scene_id}_{ep_id}"] = ep_info["description"][0]

        summary = min(ep_info["extracted_summary"], key=len)
        summary = summary.split(" owns ", 1)[1]
        # summary = "a picture of " + summary

        descr[f"{scene_id}_{ep_id}"] = summary

len(descr)

684

In [23]:
ep_info

{'episode_id': '1431',
 'scene_id': 'hm3d_v0.2/val/00877-4ok3usBNeis/4ok3usBNeis.basis.glb',
 'scene_dataset_config': './data/scene_datasets/hm3d_v0.2/hm3d_annotated_basis.scene_dataset_config.json',
 'object_category': 'cloth',
 'object_id': 'cloth_423',
 'description': ['white cloth located near the dresser and the curtain. it is near the bed on the corner of the room.',
  '',
  ''],
 'owner': 'Scarlett',
 'floor_id': '1',
 'summary': 'On the first floor, the kitchen features a stainless steel microwave situated over a white oven and near the wooden cabinets, which is shared by Elsie, Mila, and Scarlett. Nearby, Elsie also owns a double sink cabinet located next to the kitchen counter and dishwasher. The kitchen is further adorned with dark wood cabinets above a modern fridge, belonging to Elijah. In the living room, Elijah is the owner of a black leather couch placed near a lamp table and close to an old TV. Moving to the bathroom, Elsie possesses a blue and white striped towel foun

In [ ]:
descr

In [ ]:
counts = []
high_id = None
l_max = 0
for k, v in descr.items():

    l = len(v)
    counts.append( l )

    if l > l_max:
        high_id = k
        l_max = l


print(high_id, l_max, descr[high_id])

In [ ]:
print(np.mean(counts), np.std(counts))

In [ ]:
from vlfm.vlm.owlv2 import Owlv2_Detector_t


# model = Owlv2_Detector_t()

In [ ]:
q = "yes"
q = descr[high_id]

model.set_query(q)

model.is_query_in_image(img_rand)

In [ ]:
# #Check if token is acceptable

# img_rand = torch.randn((3, 640, 480))
# descr_good = {}

# for k, v in tqdm(descr.items()):

#     model.set_query(v)

#     try:
#         model.is_query_in_image(target_image = img_rand)
#         descr_good[k] = v
#     except:
#         pass


In [116]:
len(descr_good)

514

In [ ]:
len(descr_good)

In [ ]:
len(descr_good)

In [117]:
descr_good

{'ziup5kvtCCR_717': 'a wooden dining table in the living room',
 'ziup5kvtCCR_718': 'the queen-sized bed located near the window in bedroom 2',
 'ziup5kvtCCR_720': 'the queen-sized bed located near the window in bedroom 2',
 'ziup5kvtCCR_723': 'the queen-sized bed located near the window in bedroom 2',
 'ziup5kvtCCR_725': 'a queen-sized bed in bedroom 2',
 'ziup5kvtCCR_727': 'a fireplace in the living room',
 'ziup5kvtCCR_728': 'a fireplace in the living room',
 'ziup5kvtCCR_729': 'a fireplace in the living room',
 'ziup5kvtCCR_730': 'a fireplace in the living room',
 'ziup5kvtCCR_731': 'a white chair in the living room',
 'ziup5kvtCCR_734': 'a stacked washer-dryer located under the white cabinet in the laundry room',
 'ziup5kvtCCR_738': 'a wall-mounted TV in bedroom 1',
 'ziup5kvtCCR_739': 'a wall-mounted TV in bedroom 1',
 'ziup5kvtCCR_743': 'the stacked washer-dryer in the laundry room',
 'ziup5kvtCCR_747': 'the stacked washer-dryer in the laundry room',
 'ziup5kvtCCR_750': 'a sideb

### Filter PersONAL episodes : OwLv2

In [6]:
import os
import json
import gzip
import shutil
import torch
from tqdm import tqdm
from vlfm.vlm.owlv2 import Owlv2_Detector_t

try:
    if model:
        print("Model already loaded!")
    else:
        model = Owlv2_Detector_t()
        print(f"Loading model (1)...")
except:
    model = Owlv2_Detector_t()
    print(f"Loading model (2)...")


img_rand = torch.randn((3, 640, 480))

Model already loaded!


In [7]:
### ---- Utils -----

import os
import sys
from contextlib import contextmanager


def open_file(path):
    with gzip.open(path, "r") as f:
        return json.load(f)

def save_file(path, info):

    assert path.endswith(".json.gz")

    with gzip.open(path, "wt") as f:
        json.dump(info, f)

@contextmanager
def suppress_output():
    with open(os.devnull, "w") as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


In [8]:
personal_dir = "/mnt/PersONAL/data/new"
test_dir = os.path.join(personal_dir, "test_baselines")
owl_dir = os.path.join(personal_dir, "owl_baselines")
os.makedirs(owl_dir, exist_ok=True)

In [9]:
info = open_file(os.path.join(test_dir, "easy/easy.json.gz"))
info = open_file("/mnt/PersONAL/data/new/easy/easy.json.gz")
info = open_file("/mnt/PersONAL/data/new/l3mvn_baseline/easy/easy.json.gz")
# info = open_file("/mnt/PersONAL/data/new/test_baselines/easy/easy.json.gz")
info = open_file("/mnt/L3MVN/data/objectgoal_PersONAL/active_new/val/l3mvn_baseline/easy/easy.json.gz")
info.keys()

dict_keys(['episodes', 'category_to_task_category_id', 'category_to_scene_annotation_category_id'])

In [ ]:
for mode in ["easy", "medium", "hard"]:

    print(f"Mode : {medium}")
    pbar = tqdm(total = 600 if mode == "easy" else 684)

    source_dir = os.path.join(test_dir, mode, "content")
    dest_dir = os.path.join(owl_dir, mode, "content")
    os.makedirs(dest_dir, exist_ok=True)

    #Copy file like easy.json.gz
    source_mode_file = os.path.join(test_dir, mode, f"{mode}.json.gz")
    dest_mode_file = os.path.join(owl_dir, mode, f"{mode}.json.gz")
    if not os.path.exists(dest_mode_file):
        shutil.copy(source_mode_file, dest_mode_file)


    for f_scene in os.listdir(source_dir):
        
        f_scene_path = os.path.join(source_dir, f_scene)
        source_scene_info = open_file(f_scene_path)
        source_eps = len(source_scene_info["episodes"])
        scene_id = source_scene_info["episodes"][0]["scene_id"].split("/")[-1].split(".")[0]

        dest_scene_path = os.path.join(dest_dir, f_scene)
        if os.path.exists(dest_scene_path):
            print(f"Skipping scene file : {f_scene}. Already exists at destination!")
            continue

        dest_scene_info = source_scene_info.copy()
        dest_scene_info["episodes"] = []

        for ep_info in source_scene_info["episodes"]:
            
            #Extract the summary : Smallest summary -> Remove "Person owns " part of summary
            summary = min(ep_info["extracted_summary"], key=len)
            summary = summary.split(" owns ", 1)[1]

            #Check if summary is within the token threshold (try script)
            with suppress_output():

                model.set_query(summary)

                try:
                    model.is_query_in_image(target_image = img_rand)
                    dest_scene_info["episodes"].append(ep_info)
                except:
                    pass

            pbar.update()

                    
        dest_eps = len(dest_scene_info["episodes"])
        print(f"Valid Episodes for scene {scene_id} : {dest_eps} / {source_eps}")
        
        #Save scene with valid episodes
        save_file(dest_scene_path, dest_scene_info)

    pbar.close()
            

 96%|█████████▋| 579/600 [06:53<00:16,  1.28it/s]

Valid Episodes for scene VBzV5z6i1WS : 16 / 24


100%|██████████| 600/600 [07:08<00:00,  1.40it/s]


Valid Episodes for scene 4ok3usBNeis : 12 / 21
Mode : Easy


  3%|▎         | 23/684 [00:18<09:00,  1.22it/s]

...

  7%|▋         | 49/684 [00:38<08:37,  1.23it/s]

Valid Episodes for scene cvZr5TUy5C5 : 21 / 26


 10%|█         | 71/684 [00:55<08:18,  1.23it/s]

Valid Episodes for scene y9hTuugGdiq : 18 / 22


 12%|█▏        | 83/684 [01:05<08:01,  1.25it/s]